# Spatial features

Turns the cleaned tracking from `helper_preprocessing.ipynb` into per-frame evidence about which team controls the ball.
`SpatialFeatureExtractor.tabular` computes:

* the distance from the ball to the nearest home and the nearest away player, and the difference between the two;
* player counts and the home share within 2, 5 and 10 m of the ball, plus a distance-weighted home share;
* ball speed, acceleration, change of direction and sudden jumps in velocity;
* dead-ball cues: the ball outside the pitch, everyone standing still;
* team shape;
* how far to trust the ball position (observed, spline or player cluster, and the distance to the nearest real detection);
* a summary of the players within 15 m of the ball (`nbr_*`).

The home team changes ends at half time, so we measure every direction-dependent feature from the home team's side
(`*_hp`).

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"     

def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
load_notebook_defs("helper_preprocessing.ipynb", globals())

In [ ]:
def load_public_gt(path: Path, n_frames: int) -> np.ndarray:
    """evaluation/gt/BAR-ATH_possession_gt_public.csv -> (T,) '<U1' array, 'X' where there is no ground truth."""
    df = pd.read_csv(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    lab = df["label"].astype(str).str.strip().str.upper().to_numpy()
    frame = df["frame"].to_numpy(np.int64)
    if not set(np.unique(lab)) <= {"H", "A", "D", "X"}:
        raise ValueError(f"unexpected labels in {path}: {sorted(set(lab))}")
    gt = np.full(n_frames, "X", dtype="<U1")
    ok = (frame >= 0) & (frame < n_frames)
    gt[frame[ok]] = lab[ok]
    return gt

In [ ]:
@dataclass
class FeatureConfig:
    radii_m: Tuple[float, ...] = (2.0, 5.0, 10.0)      # density-ratio radii around the ball (spec)
    graph_radius_m: float = 15.0                        # radius of the ball's neighbourhood summary (spec)
    soft_sigmas_m: Tuple[float, ...] = (2.0, 5.0)       # distance-kernel widths for the soft home share
    kin_lag: int = 3                                    # frames either side for the direction-change / |dv| anomaly
    still_speed_mps: float = 1.0
    touch_dv_mps: float = 2.0                           # ball velocity change that counts as a touch ...
    touch_dist_m: float = 3.0                           # ... when a player is at most this close
    press_radius_m: float = 4.0                         # opponents this close to the ball-nearest player = pressure
    support_radius_m: float = 12.0                      # team-mates this close to him = passing options
    line_quantiles: Tuple[float, float] = (0.15, 0.85)  # 'back' and 'front' line of a team (rank among its players)


def _shift(arr: np.ndarray, k: int) -> np.ndarray:
    """arr[t - k] for k > 0, arr[t + |k|] for k < 0; NaN padded."""
    out = np.full_like(arr, np.nan)
    if k > 0:
        out[k:] = arr[:-k]
    elif k < 0:
        out[:k] = arr[-k:]
    else:
        out[:] = arr
    return out


def _masked_mean(values: np.ndarray, mask: np.ndarray):
    cnt = mask.sum(1)
    s = np.where(mask, values, 0.0).sum(1)
    return np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan), cnt


def _prev_next(mask: np.ndarray):
    """For every frame: index of the latest True at or before it, and of the earliest True at or after it (-1 / n if none)."""
    n = len(mask)
    ar = np.arange(n)
    prev = np.maximum.accumulate(np.where(mask, ar, -1))
    nxt = np.minimum.accumulate(np.where(mask, ar, n)[::-1])[::-1]
    return prev, nxt


def _take(values: np.ndarray, idx: np.ndarray, valid: np.ndarray) -> np.ndarray:
    out = np.full(len(idx), np.nan, np.float32)
    out[valid] = values[idx[valid]]
    return out


def _row_quantile(values: np.ndarray, mask: np.ndarray, q: float) -> np.ndarray:
    """Per-row quantile (nearest rank) over the masked entries; NaN when a row has none."""
    s = np.sort(np.where(mask, values, np.nan), axis=1)             # NaNs sort last
    cnt = mask.sum(1)
    k = np.clip(np.rint(q * (cnt - 1)), 0, s.shape[1] - 1).astype(int)
    return np.where(cnt > 0, s[np.arange(len(s)), k], np.nan)


class SpatialFeatureExtractor:
    """Frame-level features from a :class:`CleanTracking` ."""

    def __init__(self, cfg: Optional[FeatureConfig] = None):
        self.cfg = cfg or FeatureConfig()

    # ------------------------------------------------------------------ tabular mode
    def _block(self, c: "CleanTracking", a: int, b: int) -> Dict[str, np.ndarray]:
        cfg, fps = self.cfg, c.fps
        L, W = c.pitch_m
        sl = slice(a, b + 1)
        px, py, vx, vy = c.px[sl], c.py[sl], c.vx[sl], c.vy[sl]
        bx = c.ball_xy[sl, 0].astype(np.float32)
        by = c.ball_xy[sl, 1].astype(np.float32)
        n = len(bx)
        ar = np.arange(n)
        home, away = (c.team == 0)[None, :], (c.team == 1)[None, :]
        valid = np.isfinite(px) & np.isfinite(py)
        ball_ok = np.isfinite(bx) & np.isfinite(by)
        dxp, dyp = px - bx[:, None], py - by[:, None]
        dist = np.hypot(dxp, dyp)                              # NaN if the player or the ball is missing
        pspeed = np.hypot(vx, vy)
        mv = np.isfinite(pspeed)
        f: Dict[str, np.ndarray] = {}

        # ---- ball geometry + dead-ball cues
        f["ball_x"], f["ball_y"] = bx, by
        f["ball_goal_dist"] = np.minimum(bx, L - bx)
        f["ball_side_dist"] = np.minimum(by, W - by)
        out_d = np.hypot(np.maximum(np.maximum(-bx, bx - L), 0.0), np.maximum(np.maximum(-by, by - W), 0.0))
        f["ball_out_dist"] = out_d
        f["ball_in_pitch"] = np.where(ball_ok, (out_d <= 0.3).astype(np.float32), np.nan)
        sgn = 1.0 if np.nanmean(px[:, c.team == 0]) < np.nanmean(px[:, c.team == 1]) else -1.0   # +1: home attacks to the right
        f["ball_x_hp"] = bx if sgn > 0 else L - bx

        # ---- reliability of the ball position
        src = c.ball_src[sl]
        f["ball_observed"] = (src == 0).astype(np.float32)
        f["ball_src_spline"] = (src == 1).astype(np.float32)
        f["ball_src_linear"] = (src == 2).astype(np.float32)
        f["ball_src_cluster"] = (src == 3).astype(np.float32)
        f["ball_gap_len_log"] = np.log1p(c.ball_gap_len[sl]).astype(np.float32)
        f["ball_dist_obs"] = np.minimum(c.ball_dist_obs[sl], 250.0).astype(np.float32)

        # ---- nearest players: d_H1, d_A1, delta_d = d_H1 - d_A1 (negative = home closer)
        dH = np.where(home & np.isfinite(dist), dist, np.inf)
        dA = np.where(away & np.isfinite(dist), dist, np.inf)
        sH, sA = np.sort(dH, axis=1), np.sort(dA, axis=1)
        fin = lambda v: np.where(np.isfinite(v), v, np.nan)
        d_H1, d_H2, d_A1, d_A2 = fin(sH[:, 0]), fin(sH[:, 1]), fin(sA[:, 0]), fin(sA[:, 1])
        f.update(d_H1=d_H1, d_H2=d_H2, d_A1=d_A1, d_A2=d_A2)
        f["delta_d"] = d_H1 - d_A1
        f["delta_d2"] = d_H2 - d_A2
        f["d_min"] = np.fmin(d_H1, d_A1)
        f["nearest_is_home"] = np.where(np.isfinite(f["delta_d"]), (f["delta_d"] < 0).astype(np.float32), np.nan)

        # ---- density ratios inside r metres of the ball, and a distance-weighted share
        for r in cfg.radii_m:
            nh = ((dist < r) & home).sum(1).astype(np.float32)
            na = ((dist < r) & away).sum(1).astype(np.float32)
            f[f"n_H_{r:g}m"], f[f"n_A_{r:g}m"] = nh, na
            f[f"home_share_{r:g}m"] = np.where(nh + na > 0, nh / np.maximum(nh + na, 1), 0.5)
        for s in cfg.soft_sigmas_m:
            w = np.where(np.isfinite(dist), np.exp(-dist / s), 0.0)
            wh, wa = (w * home).sum(1), (w * away).sum(1)
            f[f"soft_home_share_{s:g}m"] = np.where(wh + wa > 1e-6, wh / np.maximum(wh + wa, 1e-9), 0.5)
            f[f"soft_mass_{s:g}m"] = wh + wa

        # ---- ball kinematics (speed, acceleration, directional anomaly)
        sbx, sby = nan_gaussian(bx, 2.0), nan_gaussian(by, 2.0)
        with np.errstate(invalid="ignore"):
            bvx, bvy = np.gradient(sbx) * fps, np.gradient(sby) * fps
            bspd = np.hypot(bvx, bvy)
            f["ball_speed"] = bspd
            f["ball_acc"] = np.gradient(bspd) * fps
            k = cfg.kin_lag
            pvx, pvy, nvx, nvy = _shift(bvx, k), _shift(bvy, k), _shift(bvx, -k), _shift(bvy, -k)
            dv = np.hypot(nvx - pvx, nvy - pvy)
            f["ball_dv"] = dv
            f["ball_anomaly"] = dv / (bspd + 1.0)
            sp_p, sp_n = np.hypot(pvx, pvy), np.hypot(nvx, nvy)
            cosang = (pvx * nvx + pvy * nvy) / np.maximum(sp_p * sp_n, 1e-6)
            ang = np.degrees(np.arccos(np.clip(cosang, -1, 1)))
            f["ball_dir_change_deg"] = np.where(np.isfinite(ang), np.where((sp_p > 1) & (sp_n > 1), ang, 0.0), np.nan)

        # ---- the nearest player of each team: how fast, and is he running onto the ball?
        for tag, idx, d1, dd in (("H", np.argmin(dH, axis=1), d_H1, dH), ("A", np.argmin(dA, axis=1), d_A1, dA)):
            ok = np.isfinite(d1)
            f[f"near{tag}_speed"] = np.where(ok, pspeed[ar, idx], np.nan)
            rx, ry = (bx - px[ar, idx]) / np.maximum(d1, 1e-3), (by - py[ar, idx]) / np.maximum(d1, 1e-3)
            f[f"near{tag}_closing"] = np.where(ok, vx[ar, idx] * rx + vy[ar, idx] * ry, np.nan)

        # ---- team shape, direction-normalised to the home team's perspective
        for tag, m in (("H", home), ("A", away)):
            mm = valid & m
            cx, cnt = _masked_mean(px, mm)
            cy, _ = _masked_mean(py, mm)
            f[f"n_tracked_{tag}"] = cnt.astype(np.float32)
            f[f"ball_to_cent_{tag}"] = np.hypot(bx - cx, by - cy)
            msq, _ = _masked_mean((px - cx[:, None]) ** 2 + (py - cy[:, None]) ** 2, mm)
            f[f"spread_{tag}"] = np.sqrt(msq)
            f[f"cent_{tag}_x_hp"] = cx if sgn > 0 else L - cx
        f["ball_minus_cent_H"] = f["ball_x_hp"] - f["cent_H_x_hp"]
        f["cent_A_minus_ball"] = f["cent_A_x_hp"] - f["ball_x_hp"]

        # ---- standing still (free kicks, restarts) vs. live play
        nmv = mv.sum(1)
        f["frac_still"] = np.where(nmv > 0, ((pspeed < cfg.still_speed_mps) & mv).sum(1) / np.maximum(nmv, 1), np.nan)
        f["mean_speed"], _ = _masked_mean(pspeed, mv)

        # ---- one round of message passing on the ball node: its neighbours within graph_radius_m
        R = cfg.graph_radius_m
        for tag, m in (("H", home), ("A", away)):
            nb = (dist < R) & m
            cnt = nb.sum(1)
            f[f"nbr_{tag}_{R:g}m"] = cnt.astype(np.float32)
            mdx, _ = _masked_mean(dxp * sgn, nb)
            mdy, _ = _masked_mean(dyp * sgn, nb)
            f[f"nbr_{tag}_dx_hp"], f[f"nbr_{tag}_dy_hp"] = np.where(cnt > 0, mdx, 0.0), np.where(cnt > 0, mdy, 0.0)
            f[f"nbr_{tag}_speed"], _ = _masked_mean(pspeed, nb & mv)

        # ---- carrying the ball, pressure and support around the nearest player of each team
        iH, iA = np.argmin(dH, axis=1), np.argmin(dA, axis=1)
        with np.errstate(invalid="ignore"):
            for tag, ii, d1, own, opp in (("H", iH, d_H1, home, away), ("A", iA, d_A1, away, home)):
                ok = np.isfinite(d1)
                f[f"near{tag}_rel_speed"] = np.where(ok, np.hypot(bvx - vx[ar, ii], bvy - vy[ar, ii]), np.nan)   # ~0 = ball travels with him
                dd_ = np.hypot(px - px[ar, ii][:, None], py - py[ar, ii][:, None])      # from that player to everybody
                f[f"press_on_{tag}1"] = np.where(ok, ((dd_ < cfg.press_radius_m) & opp).sum(1), np.nan)
                f[f"support_{tag}1"] = np.where(ok, np.maximum(((dd_ < cfg.support_radius_m) & own).sum(1) - 1, 0), np.nan)

        # ---- team-wide movement and shape, each team in its own attacking direction (0 = own goal line)
        pxh = px if sgn > 0 else L - px                                   # home perspective: home attacks towards larger values
        lo_q, hi_q = cfg.line_quantiles
        for tag, m, fwd in (("H", home, sgn), ("A", away, -sgn)):
            mm, mmv = valid & m, mv & m
            f[f"team_fwd_vel_{tag}"], _ = _masked_mean(vx * fwd, mmv)         # >0: the whole team is moving towards the opponent's goal
            f[f"team_speed_{tag}"], _ = _masked_mean(pspeed, mmv)
            depth = pxh if tag == "H" else L - pxh
            f[f"line_back_{tag}"] = _row_quantile(depth, mm, lo_q)
            f[f"line_front_{tag}"] = _row_quantile(depth, mm, hi_q)
            f[f"length_{tag}"] = f[f"line_front_{tag}"] - f[f"line_back_{tag}"]
            f[f"width_{tag}"] = _row_quantile(py, mm, hi_q) - _row_quantile(py, mm, lo_q)
        bxh = f["ball_x_hp"]
        with np.errstate(invalid="ignore"):
            f["H_ahead_of_ball"] = np.where(ball_ok, ((pxh > bxh[:, None]) & valid & home).sum(1), np.nan)
            f["A_ahead_of_ball"] = np.where(ball_ok, ((pxh < bxh[:, None]) & valid & away).sum(1), np.nan)

        # ---- context of a missing-ball stretch: what the evidence looked like at the last / next real ball detection
        with np.errstate(invalid="ignore"):
            obs = (src == 0) & np.isfinite(f["delta_d"])
        prv, nxt = _prev_next(obs)
        has_p, has_n = prv >= 0, nxt < n
        f["t_since_obs_log"] = np.where(has_p, np.log1p(ar - prv), np.nan)
        f["t_to_obs_log"] = np.where(has_n, np.log1p(nxt - ar), np.nan)
        span = np.where(has_p & has_n, nxt - prv, 0)
        f["gap_progress"] = np.where(span > 0, (ar - prv) / np.maximum(span, 1), np.nan)      # 0 at the start of a gap, 1 at its end
        hs_ = f[f"soft_home_share_{cfg.soft_sigmas_m[-1]:g}m"]
        for name, val in (("dd", f["delta_d"]), ("hs", hs_)):
            f[f"{name}_prev_obs"] = _take(val, prv, has_p)
            f[f"{name}_next_obs"] = _take(val, nxt, has_n)
        bxh_obs = np.where(src == 0, bxh, np.nan)
        f["gap_dx_hp"] = np.where(has_p & has_n, _take(bxh_obs, nxt, has_n) - _take(bxh_obs, prv, has_p), np.nan)   # how far play moved during the gap

        # ---- last / next touch: a sharp change of the ball's velocity with a player right there. Possession tends to stay with the toucher.
        with np.errstate(invalid="ignore"):
            touch = (src == 0) & (f["ball_dv"] > cfg.touch_dv_mps) & (f["d_min"] < cfg.touch_dist_m)
        tp, tn = _prev_next(touch)
        has_tp, has_tn = tp >= 0, tn < n
        f["last_touch_home"] = _take(f["nearest_is_home"], tp, has_tp)
        f["t_since_touch_log"] = np.where(has_tp, np.log1p(ar - tp), np.nan)
        f["next_touch_home"] = _take(f["nearest_is_home"], tn, has_tn)
        f["t_to_touch_log"] = np.where(has_tn, np.log1p(tn - ar), np.nan)
        return {k: np.asarray(v, np.float32) for k, v in f.items()}

    def tabular(self, c: "CleanTracking") -> pd.DataFrame:
        """(T, F) float32 DataFrame indexed by video frame. Rows outside the graded halves are NaN."""
        T = c.n_frames
        blocks = [(a, b, self._block(c, a, b)) for _, a, b in c.half_blocks]
        names = list(blocks[0][2].keys())
        data = np.full((T, len(names)), np.nan, np.float32)
        for a, b, f in blocks:
            for j, name in enumerate(names):
                data[a:b + 1, j] = f[name]
        return pd.DataFrame(data, columns=names)

In [ ]:
@dataclass
class MatchInputs:
    raw: TrackingData
    gt: np.ndarray                      # (T,) '<U1', 'X' = no ground truth. Used to evaluate only, never as training data.


def load_match_inputs(gt_path: Optional[Path] = None, verbose: bool = True) -> MatchInputs:
    """Load the target match's tracking and a ground truth: ``gt_path`` = any ``frame,label`` CSV (any length, X = not covered),
    None = the public ground truth shipped in evaluation/gt/. Raises FileNotFoundError if a file is missing."""
    paths = get_paths()
    if gt_path is not None:
        paths["gt_public"] = Path(gt_path)
    require_files(paths, ("atd_meta", "atd_raw", "gt_public"))
    raw = load_atd_tracking(paths, verbose=verbose)
    return MatchInputs(raw=raw, gt=load_public_gt(paths["gt_public"], raw.n_frames))